# 01 — Data exploration

Exploration and data quality checks for NYC Yellow Taxi trips (Jan–Mar 2024).
This notebook is a one-off investigation; recurring checks become part of the pipeline in stage 3.

In [8]:
# Imports

import sys
import pandas as pd
from pathlib import Path

In [9]:
# Project root

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Making src/ importable and starting Spark

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from spark_session import get_spark
spark = get_spark()

## 2.1 Schema comparison across months

TLC warns that schemas may change between periods. Each month is read separately before any union.

In [27]:
month_files = sorted(RAW_DIR.glob("*.parquet"))

column_list = []
parquet_name = []
column_size_by_file = []
for path in month_files:
    trips_month = spark.read.parquet(str(path))
    column_list.append(trips_month.dtypes)
    column_size_by_file.append(len(trips_month.columns))
    parquet_name.append(path.stem)

if len(set(column_size_by_file)) != 1:
    raise ValueError("Files have different number of columns.")
else:
    print("All files have the same number of columns.")
    column_size = column_size_by_file[0]


All files have the same number of columns.


In [ ]:
df_file_columns = pd.DataFrame(data=column_list, columns=range(0,column_size),
                         index=parquet_name)

# Checking existence of columns with different names or data_types
has_nonunique_col = False
nonunique_cols = []
for column in range(0, column_size):
    if (not(df_file_columns[column].nunique())):
        has_nonunique_col = True
        nonunique_cols.append(column)

if has_nonunique_col:
    print(f"Non unique cols: {nonunique_cols}")
else:
    print(f"All files have the same column names and data types.")



All files have the same column names and data types.


**Result:** the three files have identical schemas: 19 columns with the same names, types and order.
They can be read together without adjustments. The expected schema is documented in
`docs/data_dictionary.md` and will be checked automatically by the build in stage 3.

## 2.2 Data profile

Now we will read all three months together as one table.

In [36]:
# Read the monthly Parquet files found by glob (no wildcard warning)
trips_raw = spark.read.parquet(*[str(path) for path in month_files])

# Creates temporary view
trips_raw.createOrReplaceTempView('trips_raw')

# Checks if everything worked fine
result = spark.sql("SELECT * FROM trips_raw LIMIT 5")
# Shows completes values even if they are too long
result.show(truncate=False)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

## 2.1 Schema comparison across months

The TLC warns that file schemas may change between periods (column names, types or order).
Before combining January, February and March into a single table, each monthly file is read
separately and its schema is compared position by position: column name and data type.

If any difference appears, it must be handled before the union; otherwise the files can be
read together as one table.

In [40]:
# Counting registers by month and year

result = spark.sql("""
SELECT EXTRACT(MONTH FROM tpep_pickup_datetime) AS month, EXTRACT(YEAR FROM tpep_pickup_datetime) AS year, COUNT(*) AS trips
FROM trips_raw
GROUP BY EXTRACT(MONTH FROM tpep_pickup_datetime), EXTRACT(YEAR FROM tpep_pickup_datetime)
ORDER BY year DESC, month ASC
""")

result.show()


+-----+----+-------+
|month|year|  trips|
+-----+----+-------+
|    1|2024|2964617|
|    2|2024|3007533|
|    3|2024|3582607|
|    4|2024|      2|
|   12|2023|     10|
|    1|2009|      4|
|   12|2008|      1|
|   12|2002|      4|
+-----+----+-------+



- **Volume grows month over month**, even after adjusting for the number of days
  (+8% Jan→Feb, +11% Feb→Mar in trips per day). This suggests seasonality
  (lower demand in winter) and is worth exploring in the analysis stage.
- **21 records have pickup dates outside Jan–Mar 2024** (0.0002% of the data):
  - Dec 2023 (10) and Apr 2024 (2): likely trips around the period boundaries (January and April, respectively).
  - 2002, 2008 and 2009 (9): clearly invalid dates, probably from taximeter clock errors.
- These records are investigated in 2.3 a; the handling rule is defined in 2.4.


## 2.2 Data profiling

In [64]:
# Gets null values count for each column

# Generates a SQL string getting null values in each column
str_sql = "SELECT "
for column_name in trips_raw.columns:
    str_sql += f"COUNT(*) - COUNT({column_name}) AS {column_name},\n"

str_sql = str_sql[:len(str_sql)-2] + "\nFROM trips_raw"
print(str_sql)



SELECT COUNT(*) - COUNT(VendorID) AS VendorID,
COUNT(*) - COUNT(tpep_pickup_datetime) AS tpep_pickup_datetime,
COUNT(*) - COUNT(tpep_dropoff_datetime) AS tpep_dropoff_datetime,
COUNT(*) - COUNT(passenger_count) AS passenger_count,
COUNT(*) - COUNT(trip_distance) AS trip_distance,
COUNT(*) - COUNT(RatecodeID) AS RatecodeID,
COUNT(*) - COUNT(store_and_fwd_flag) AS store_and_fwd_flag,
COUNT(*) - COUNT(PULocationID) AS PULocationID,
COUNT(*) - COUNT(DOLocationID) AS DOLocationID,
COUNT(*) - COUNT(payment_type) AS payment_type,
COUNT(*) - COUNT(fare_amount) AS fare_amount,
COUNT(*) - COUNT(extra) AS extra,
COUNT(*) - COUNT(mta_tax) AS mta_tax,
COUNT(*) - COUNT(tip_amount) AS tip_amount,
COUNT(*) - COUNT(tolls_amount) AS tolls_amount,
COUNT(*) - COUNT(improvement_surcharge) AS improvement_surcharge,
COUNT(*) - COUNT(total_amount) AS total_amount,
COUNT(*) - COUNT(congestion_surcharge) AS congestion_surcharge,
COUNT(*) - COUNT(Airport_fee) AS Airport_fee
FROM trips_raw


In [66]:
# SQL Query

spark.sql(str_sql).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         751962|            0|    751962|            751962|           0|           0|           0|          0|    0|      0|         

In [67]:
spark.sql("""
SELECT COUNT(*) AS rows_with_all_five_null
FROM trips_raw
WHERE passenger_count IS NULL
  AND RatecodeID IS NULL
  AND store_and_fwd_flag IS NULL
  AND congestion_surcharge IS NULL
  AND Airport_fee IS NULL
""").show()

+-----------------------+
|rows_with_all_five_null|
+-----------------------+
|                 751962|
+-----------------------+



In [68]:
spark.sql("""
SELECT payment_type, VendorID, COUNT(*) AS trips
FROM trips_raw
WHERE passenger_count IS NULL
GROUP BY payment_type, VendorID
ORDER BY trips DESC
""").show()

+------------+--------+------+
|payment_type|VendorID| trips|
+------------+--------+------+
|           0|       2|542016|
|           0|       1|209226|
|           0|       6|   720|
+------------+--------+------+



In [69]:
spark.sql("""
SELECT COUNT(*) AS flex_fare_trips
FROM trips_raw
WHERE payment_type = 0
""").show()

+---------------+
|flex_fare_trips|
+---------------+
|         751962|
+---------------+



**Result:** only five columns have nulls, and all with the same count: 751,962 rows (7.9%).

- The nulls are always in the **same rows**: `passenger_count`, `RatecodeID`,
  `store_and_fwd_flag`, `congestion_surcharge` and `Airport_fee` are missing together.
- These rows are **exactly the Flex Fare trips** (`payment_type = 0`): every Flex Fare trip
  has the five fields null, and no other trip has any null.
- Flex Fare is TLC's upfront-pricing program: the passenger books through an e-hail app and
  accepts a binding fare before the trip, paid electronically in the app. The taximeter
  rate fields are therefore not recorded.
- They come from three vendors (Curb, Creative Mobile, Myle), so this is a property of the
  trip type, not a device failure.
- Fare, distance, timestamps and zones are filled, so these look like real trips.
  Removing them would drop 7.9% of the volume; the handling rule is defined in 2.4.